# Step 4C (Phase 1): Frozen sPCI Test Split Evaluation & Blind Pair Scoring
**Objective**: Compute predictions, features, and sPCI on the test splits and test pair images strictly **WITHOUT** loading or inspecting `labels/pairs_human.csv`.
- **Model**: ResNet-34 U-Net (Final Naive Run B frozen weights)
- **Parameters**: Frozen under git tag `spci-v1-frozen`
- **Output Artifacts**:
  1. `results/spci_test_pairs.csv` (120 test images across 60 pairs)
  2. Full test splits evaluation: Spearman $\rho$ and MAE vs GT

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import os
import sys
import subprocess
import shutil
import time
import json
from pathlib import Path

print("Installing required dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8", "scipy==1.17.1", "pyyaml==6.0.2"], check=True)

# Clone repository
repo_dir = Path("/kaggle/working/repo")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print("Cloning repository from GitHub...")
subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

import torch
import torchvision
import numpy as np
import pandas as pd
import cv2
import yaml
from PIL import Image
from scipy.stats import spearmanr
import segmentation_models_pytorch as smp

print("=" * 80)
print("ENVIRONMENT INFORMATION:")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • PyTorch:      {torch.__version__}")
print(f"  • Torchvision:  {torchvision.__version__}")
print(f"  • SMP Version:  {smp.__version__}")
print(f"  • CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  • GPU Device:   {torch.cuda.get_device_name(0)}")
print("=" * 80)

In [ ]:
# ==============================================================================
# 2. RESOLVE DATASET ROOTS & VERIFY WEIGHTS
# ==============================================================================
import hashlib
from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.pavement_features import compute_features, rescale_mask
from src.data_rules import load_standardized_mask
from src.datasets import RoadDistressDataset
from src.spci import compute_spci, load_spci_config, get_condition_class

dataset_roots = {}
for src in ['crack500', 'road_crack', 'pothole']:
    meta = DATASET_SPECS[src]
    resolved = resolve_dataset_root(meta['owner'], meta['name'])
    if resolved is None:
        raise FileNotFoundError(f"CRITICAL: Failed to locate dataset root for {src}")
    dataset_roots[src] = resolved
    print(f"  • {src:<12} -> {resolved}")

weights_candidates = [
    Path("/kaggle/input/datasets/vedshah04/road-pci-seg-final-weights/best.pt"),
    Path("/kaggle/input/road-pci-seg-final-weights/best.pt"),
    repo_dir / "scratch/downloaded_final_weights/best.pt"
]
weights_path = None
for p in weights_candidates:
    if p.exists():
        weights_path = p
        break

if weights_path is None:
    raise FileNotFoundError("CRITICAL: Failed to locate frozen weights best.pt")

hasher = hashlib.sha256()
with open(weights_path, 'rb') as f:
    while chunk := f.read(65536):
        hasher.update(chunk)
sha256 = hasher.hexdigest()

EXPECTED_SHA256 = "9269c5edb6e5d37b359bd3337fbd83c60901cd2b241f9ef789e9116fee400ef7"
print(f"[OK] Weights located at: {weights_path}")
print(f"  • SHA256: {sha256}")
assert sha256 == EXPECTED_SHA256, f"Weight SHA256 mismatch! Expected {EXPECTED_SHA256}, got {sha256}"

In [ ]:
# ==============================================================================
# 3. VERIFY FROZEN sPCI CONFIGURATION (TAG: spci-v1-frozen)
# ==============================================================================
spci_cfg = load_spci_config(repo_dir / "configs/spci_params.yaml")

print("=" * 80)
print("FROZEN sPCI CONFIGURATION VERIFICATION:")
print(f"  • Crack Width T_low:        {spci_cfg['crack_severity']['low_threshold_px']} px")
print(f"  • Crack Width T_high:       {spci_cfg['crack_severity']['high_threshold_px']} px")
print(f"  • Network Reference n_ref:  {spci_cfg['network_factor']['n_ref']}")
print(f"  • Network Beta:             {spci_cfg['network_factor']['beta']}")
print(f"  • Noise Floor Tau:          {spci_cfg.get('noise_floor', {}).get('tau_short_sides', 0.0)}")
print(f"  • Rating Bands (ASTM D6433): {spci_cfg['condition_classes']}")
print("=" * 80)

assert spci_cfg['crack_severity']['low_threshold_px'] == 4.394
assert spci_cfg['crack_severity']['high_threshold_px'] == 11.588
assert spci_cfg['network_factor']['n_ref'] == 1.275
assert spci_cfg['network_factor']['beta'] == 0.5
assert spci_cfg.get('noise_floor', {}).get('tau_short_sides', 0.0) == 0.0
print("[PASS] Verified that active configuration strictly matches spci-v1-frozen.")

In [ ]:
# ==============================================================================
# 4. FULL TEST SPLIT INFERENCE & SCORING (N = 200)
# ==============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Inference device: {device}")

model = smp.Unet(encoder_name='resnet34', classes=2)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

manifest_path = repo_dir / "manifests/segmentation_manifest.csv"
manifest_df = pd.read_csv(manifest_path)
test_manifest_df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)

test_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='test', crop_size=512)
print(f"Full Test Dataset: {len(test_dataset)} images across sources: {test_manifest_df['source'].value_counts().to_dict()}")

t0_test = time.time()
test_cache = {}

with torch.no_grad():
    for idx in range(len(test_dataset)):
        image_t, target_t, label_mask, metas = test_dataset[idx]
        image_b = image_t.unsqueeze(0).to(device)
        src = metas['source']
        unpad_info = metas['eval_unpad']
        y0, x0, h, w = unpad_info
        
        # Forward Flip-TTA on GPU
        probs_orig = torch.sigmoid(model(image_b))
        probs_h = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-1]))), dims=[-1])
        probs_v = torch.flip(torch.sigmoid(model(torch.flip(image_b, dims=[-2]))), dims=[-2])
        probs_tta = ((probs_orig + probs_h + probs_v) / 3.0).cpu()
        
        p_crack = (probs_tta[0, 0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        p_pot = (probs_tta[0, 1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        
        # Ground-truth masks (for pothole images, GT has only the pothole channel)
        t_crack = (target_t[0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[0] else None
        t_pot = (target_t[1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[1] else None
        
        # Compute features
        feats_pred = compute_features(p_crack, p_pot, prune_branches=True, min_branch_len=12)
        feats_gt = compute_features(t_crack, t_pot, prune_branches=True, min_branch_len=12)
        
        # Compute sPCI
        spci_pred = compute_spci(p_crack, p_pot, config=spci_cfg)
        spci_gt = compute_spci(t_crack, t_pot, config=spci_cfg)
        
        img_rel_path = metas['image_path']
        test_cache[img_rel_path] = {
            'source': src,
            'image_path': img_rel_path,
            'feats_pred': feats_pred,
            'feats_gt': feats_gt,
            'spci_pred': spci_pred,
            'spci_gt': spci_gt
        }
        
        if (idx + 1) % 50 == 0 or (idx + 1) == len(test_dataset):
            print(f"  Processed {idx + 1} / {len(test_dataset)} test images in {time.time()-t0_test:.1f}s")

print(f"[OK] Completed evaluation on all {len(test_cache)} test images in {time.time()-t0_test:.2f}s")

In [ ]:
# ==============================================================================
# 5. FULL TEST SPLIT GT vs. PREDICTED sPCI CORRELATION & ERROR
# ==============================================================================
print("=" * 80)
print("FULL TEST SPLIT EVALUATION (CRACK500: 72, ROAD_CRACK: 68, POTHOLE: 60)")
print("=" * 80)
print(f"{'Source':<14} | {'N Images':<10} | {'Spearman rho':<14} | {'MAE (Points)':<14}")
print("-" * 80)

full_test_results = {}
for src in ['crack500', 'road_crack', 'pothole']:
    src_items = [v for v in test_cache.values() if v['source'] == src]
    n_src = len(src_items)
    
    gt_scores = np.array([item['spci_gt']['spci'] for item in src_items])
    pred_scores = np.array([item['spci_pred']['spci'] for item in src_items])
    
    rho = float(spearmanr(gt_scores, pred_scores).statistic)
    mae = float(np.mean(np.abs(gt_scores - pred_scores)))
    
    full_test_results[src] = {
        'n': n_src,
        'spearman_rho': rho,
        'mae': mae
    }
    print(f"{src:<14} | {n_src:<10} | {rho:14.4f} | {mae:14.2f}")
print("=" * 80)
print("Note: For pothole images, the ground-truth annotations contain only the pothole channel.")

In [ ]:
# ==============================================================================
# 6. GENERATE results/spci_test_pairs.csv (120 PAIR IMAGES, STRICTLY WITHOUT LABELS)
# ==============================================================================
# We read ONLY labels/pairs_template.csv to extract pair_id and image filenames
template_path = repo_dir / "labels/pairs_template.csv"
pairs_template = pd.read_csv(template_path)
print(f"Loaded pairs template: {len(pairs_template)} pairs")

pair_records = []
for idx, row in pairs_template.iterrows():
    p_id = row['pair_id']
    img_a = row['image_a']
    img_b = row['image_b']
    
    if img_a not in test_cache:
        raise KeyError(f"Image A not found in test cache: {img_a}")
    if img_b not in test_cache:
        raise KeyError(f"Image B not found in test cache: {img_b}")
        
    res_a = test_cache[img_a]
    res_b = test_cache[img_b]
    
    # Record Image A
    pair_records.append({
        'pair_id': p_id,
        'which': 'A',
        'source': res_a['source'],
        'image_path': img_a,
        'sPCI': res_a['spci_pred']['spci'],
        'condition_band': res_a['spci_pred']['condition_class'],
        'crack_area_frac': res_a['feats_pred']['crack_area_frac'],
        'pothole_area_frac': res_a['feats_pred']['pothole_area_frac'],
        'crack_length_short_sides': res_a['feats_pred']['crack_length_short_sides'],
        'sPCI_gt': res_a['spci_gt']['spci'],
        'condition_band_gt': res_a['spci_gt']['condition_class'],
    })
    
    # Record Image B
    pair_records.append({
        'pair_id': p_id,
        'which': 'B',
        'source': res_b['source'],
        'image_path': img_b,
        'sPCI': res_b['spci_pred']['spci'],
        'condition_band': res_b['spci_pred']['condition_class'],
        'crack_area_frac': res_b['feats_pred']['crack_area_frac'],
        'pothole_area_frac': res_b['feats_pred']['pothole_area_frac'],
        'crack_length_short_sides': res_b['feats_pred']['crack_length_short_sides'],
        'sPCI_gt': res_b['spci_gt']['spci'],
        'condition_band_gt': res_b['spci_gt']['condition_class'],
    })

spci_pairs_df = pd.DataFrame(pair_records)
print(f"Constructed spci_test_pairs DataFrame: {len(spci_pairs_df)} rows")
assert len(spci_pairs_df) == 120

out_dir = repo_dir / "results"
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "spci_test_pairs.csv"
spci_pairs_df.to_csv(out_csv, index=False)
spci_pairs_df.to_csv(Path("/kaggle/working/spci_test_pairs.csv"), index=False)
print(f"[OK] Saved {out_csv}")

# Compute and print SHA-256 of the CSV
csv_hasher = hashlib.sha256()
with open(out_csv, 'rb') as f:
    while chunk := f.read(65536):
        csv_hasher.update(chunk)
csv_sha256 = csv_hasher.hexdigest()

print("=" * 80)
print(f"SHA-256 OF results/spci_test_pairs.csv: {csv_sha256}")
print("=" * 80)

In [ ]:
# ==============================================================================
# 7. EXPORT PHASE 1 SUMMARY JSON
# ==============================================================================
summary_payload = {
    'weights_sha256': sha256,
    'full_test_eval': full_test_results,
    'csv_sha256': csv_sha256,
    'n_pairs': len(pairs_template),
    'n_images': len(spci_pairs_df),
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open("/kaggle/working/step_4c_phase1_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, indent=2)
print("[OK] Exported /kaggle/working/step_4c_phase1_summary.json successfully.")